# ISLES'24 NCCT baseline recovery notebook

Run **Runtime → Run all**. This notebook is intentionally narrower than the previous one: it prepares only the first predeclared **NCCT-only** baseline.

Every important message and subprocess output is also written to Google Drive at `HybridStrokeSeg/ISLES2024/results/ncct_prepare.log`, so if Colab disconnects or errors, the log remains available for inspection.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, re, shutil, subprocess, tempfile, time

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
ARCHIVE = ROOT / 'raw' / 'train.7z'
SOURCE = ROOT / 'ncct_source'
RESULTS = ROOT / 'results'
NNUNET_RAW = ROOT / 'nnUNet_raw'
LOG = RESULTS / 'ncct_prepare.log'
REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')
RESULTS.mkdir(parents=True, exist_ok=True)
NNUNET_RAW.mkdir(parents=True, exist_ok=True)

def log(msg):
    text = str(msg)
    print(text, flush=True)
    with LOG.open('a', encoding='utf-8') as h:
        h.write(text + '\n')

def run_logged(cmd, **kwargs):
    log('$ ' + ' '.join(map(str, cmd)))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kwargs)
    with LOG.open('a', encoding='utf-8') as h:
        for line in p.stdout:
            print(line, end='')
            h.write(line)
    rc = p.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

LOG.write_text('ISLES24 NCCT preparation started\n', encoding='utf-8')
if not ARCHIVE.is_file():
    raise FileNotFoundError(ARCHIVE)
log(f'Archive size: {ARCHIVE.stat().st_size} bytes')


## Install tools and latest repository snapshot


In [ ]:
run_logged(['apt-get','update','-qq'])
run_logged(['apt-get','install','-y','-qq','p7zip-full','unzip'])
if REPO.exists(): shutil.rmtree(REPO)
if REPO_ZIP.exists(): REPO_ZIP.unlink()
run_logged(['curl','-fL','--retry','20','--retry-delay','3','--retry-all-errors',
            'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
            '-o',str(REPO_ZIP)])
run_logged(['unzip','-q',str(REPO_ZIP),'-d','/content'])
os.chdir(REPO)
run_logged(['python','-m','pip','install','-q','-e','.'])
log(f'Repository ready: {REPO}')


## Discover and extract exactly 149 NCCT volumes + 149 final-infarct masks

Extraction is resumable in 10-subject batches. No CTA, perfusion, raw CTP, follow-up MRI, or outcome variables are extracted in this first baseline.


In [ ]:
seven = shutil.which('7zz') or shutil.which('7z')
if not seven: raise RuntimeError('7-Zip was not installed')
log('Listing archive members. This can take several minutes on Drive...')
proc = subprocess.run([seven,'l','-slt',str(ARCHIVE)],capture_output=True,text=True,errors='replace',check=True)
with LOG.open('a',encoding='utf-8') as h: h.write('\n--- 7z listing completed ---\n')
members = [line[7:].strip().replace('\\','/') for line in proc.stdout.splitlines() if line.startswith('Path = ')]
ncct_re = re.compile(r'/(?:rawdata|raw_data)/(sub-[^/]+)/ses-0*1/[^/]*_ncct\\.nii\\.gz
mask_re = re.compile(r'/derivatives/(sub-[^/]+)/ses-0*2/[^/]*_lesion-msk\\.nii\\.gz
ncct, masks = {}, {}
for member in members:
    p='/' + member.lstrip('./')
    m=ncct_re.search(p)
    if m: ncct[m.group(1).lower()] = member
    m=mask_re.search(p)
    if m: masks[m.group(1).lower()] = member
shared=sorted(set(ncct)&set(masks))
log(f'NCCT found: {len(ncct)}; masks found: {len(masks)}; paired: {len(shared)}')\nlog('Expected actual Zenodo v7 session labels include ses-01 and ses-02; regex also accepts ses-0001/ses-0002.')
if len(shared)!=149 or len(ncct)!=149 or len(masks)!=149:
    interesting=[m for m in members if 'ncct' in m.lower() or 'lesion' in m.lower()]
    log('First matching archive paths for diagnosis:')
    for p in interesting[:80]: log('  '+p)
    raise RuntimeError(f'Expected 149 paired NCCT/mask cases, got NCCT={len(ncct)}, masks={len(masks)}, paired={len(shared)}. Persistent log: {LOG}')

state=SOURCE/'.extract_state'; state.mkdir(parents=True,exist_ok=True)
batches=[shared[i:i+10] for i in range(0,len(shared),10)]
for bi,subjects in enumerate(batches):
    marker=state/f'batch_{bi:03d}.done'
    if marker.exists():
        log(f'[{bi+1}/{len(batches)}] skip completed batch')
        continue
    selected=[]
    for sid in subjects: selected += [ncct[sid],masks[sid]]
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',suffix='.txt',delete=False) as h:
        listfile=Path(h.name); h.write('\n'.join(selected)+'\n')
    try:
        run_logged([seven,'x',str(ARCHIVE),f'-o{SOURCE}','-aoa','-scsUTF-8',f'-i@{listfile}'])
    finally:
        listfile.unlink(missing_ok=True)
    marker.write_text('\n'.join(subjects)+'\n',encoding='utf-8')
    log(f'[{bi+1}/{len(batches)}] completed {len(subjects)} subjects')
log('NCCT/mask extraction complete')
,re.I)
mask_re = re.compile(r'/derivatives/(sub-[^/]+)/ses-0002/[^/]*_lesion-msk\\.nii\\.gz$',re.I)
ncct, masks = {}, {}
for member in members:
    p='/' + member.lstrip('./')
    m=ncct_re.search(p)
    if m: ncct[m.group(1).lower()] = member
    m=mask_re.search(p)
    if m: masks[m.group(1).lower()] = member
shared=sorted(set(ncct)&set(masks))
log(f'NCCT found: {len(ncct)}; masks found: {len(masks)}; paired: {len(shared)}')
if len(shared)!=149 or len(ncct)!=149 or len(masks)!=149:
    interesting=[m for m in members if 'ncct' in m.lower() or 'lesion' in m.lower()]
    log('First matching archive paths for diagnosis:')
    for p in interesting[:80]: log('  '+p)
    raise RuntimeError(f'Expected 149 paired NCCT/mask cases, got NCCT={len(ncct)}, masks={len(masks)}, paired={len(shared)}. Persistent log: {LOG}')

state=SOURCE/'.extract_state'; state.mkdir(parents=True,exist_ok=True)
batches=[shared[i:i+10] for i in range(0,len(shared),10)]
for bi,subjects in enumerate(batches):
    marker=state/f'batch_{bi:03d}.done'
    if marker.exists():
        log(f'[{bi+1}/{len(batches)}] skip completed batch')
        continue
    selected=[]
    for sid in subjects: selected += [ncct[sid],masks[sid]]
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',suffix='.txt',delete=False) as h:
        listfile=Path(h.name); h.write('\n'.join(selected)+'\n')
    try:
        run_logged([seven,'x',str(ARCHIVE),f'-o{SOURCE}','-aoa','-scsUTF-8',f'-i@{listfile}'])
    finally:
        listfile.unlink(missing_ok=True)
    marker.write_text('\n'.join(subjects)+'\n',encoding='utf-8')
    log(f'[{bi+1}/{len(batches)}] completed {len(subjects)} subjects')
log('NCCT/mask extraction complete')
,re.I)
ncct, masks = {}, {}
for member in members:
    p='/' + member.lstrip('./')
    m=ncct_re.search(p)
    if m: ncct[m.group(1).lower()] = member
    m=mask_re.search(p)
    if m: masks[m.group(1).lower()] = member
shared=sorted(set(ncct)&set(masks))
log(f'NCCT found: {len(ncct)}; masks found: {len(masks)}; paired: {len(shared)}')
if len(shared)!=149 or len(ncct)!=149 or len(masks)!=149:
    interesting=[m for m in members if 'ncct' in m.lower() or 'lesion' in m.lower()]
    log('First matching archive paths for diagnosis:')
    for p in interesting[:80]: log('  '+p)
    raise RuntimeError(f'Expected 149 paired NCCT/mask cases, got NCCT={len(ncct)}, masks={len(masks)}, paired={len(shared)}. Persistent log: {LOG}')

state=SOURCE/'.extract_state'; state.mkdir(parents=True,exist_ok=True)
batches=[shared[i:i+10] for i in range(0,len(shared),10)]
for bi,subjects in enumerate(batches):
    marker=state/f'batch_{bi:03d}.done'
    if marker.exists():
        log(f'[{bi+1}/{len(batches)}] skip completed batch')
        continue
    selected=[]
    for sid in subjects: selected += [ncct[sid],masks[sid]]
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',suffix='.txt',delete=False) as h:
        listfile=Path(h.name); h.write('\n'.join(selected)+'\n')
    try:
        run_logged([seven,'x',str(ARCHIVE),f'-o{SOURCE}','-aoa','-scsUTF-8',f'-i@{listfile}'])
    finally:
        listfile.unlink(missing_ok=True)
    marker.write_text('\n'.join(subjects)+'\n',encoding='utf-8')
    log(f'[{bi+1}/{len(batches)}] completed {len(subjects)} subjects')
log('NCCT/mask extraction complete')
,re.I)
mask_re = re.compile(r'/derivatives/(sub-[^/]+)/ses-0002/[^/]*_lesion-msk\\.nii\\.gz$',re.I)
ncct, masks = {}, {}
for member in members:
    p='/' + member.lstrip('./')
    m=ncct_re.search(p)
    if m: ncct[m.group(1).lower()] = member
    m=mask_re.search(p)
    if m: masks[m.group(1).lower()] = member
shared=sorted(set(ncct)&set(masks))
log(f'NCCT found: {len(ncct)}; masks found: {len(masks)}; paired: {len(shared)}')
if len(shared)!=149 or len(ncct)!=149 or len(masks)!=149:
    interesting=[m for m in members if 'ncct' in m.lower() or 'lesion' in m.lower()]
    log('First matching archive paths for diagnosis:')
    for p in interesting[:80]: log('  '+p)
    raise RuntimeError(f'Expected 149 paired NCCT/mask cases, got NCCT={len(ncct)}, masks={len(masks)}, paired={len(shared)}. Persistent log: {LOG}')

state=SOURCE/'.extract_state'; state.mkdir(parents=True,exist_ok=True)
batches=[shared[i:i+10] for i in range(0,len(shared),10)]
for bi,subjects in enumerate(batches):
    marker=state/f'batch_{bi:03d}.done'
    if marker.exists():
        log(f'[{bi+1}/{len(batches)}] skip completed batch')
        continue
    selected=[]
    for sid in subjects: selected += [ncct[sid],masks[sid]]
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',suffix='.txt',delete=False) as h:
        listfile=Path(h.name); h.write('\n'.join(selected)+'\n')
    try:
        run_logged([seven,'x',str(ARCHIVE),f'-o{SOURCE}','-aoa','-scsUTF-8',f'-i@{listfile}'])
    finally:
        listfile.unlink(missing_ok=True)
    marker.write_text('\n'.join(subjects)+'\n',encoding='utf-8')
    log(f'[{bi+1}/{len(batches)}] completed {len(subjects)} subjects')
log('NCCT/mask extraction complete')


## Audit the real 149-case NCCT subset


In [ ]:
AUDIT = RESULTS/'isles2024_ncct_audit.json'
run_logged(['python','scripts/audit_isles2024.py',str(SOURCE),'--channels','ncct','--json',str(AUDIT)])
log(f'Audit saved: {AUDIT}')


## Stage persistent NCCT-only nnU-Net Dataset504


In [ ]:
run_logged(['python','scripts/stage_isles2024_nnunet.py',str(SOURCE),str(NNUNET_RAW),
            '--dataset-id','504','--dataset-name','ISLES2024_NCCT','--channels','ncct',
            '--mode','copy','--resume','--split-seed','2026'])
DATASET=NNUNET_RAW/'Dataset504_ISLES2024_NCCT'
log(f'imagesTr: {len(list((DATASET/"imagesTr").glob("*.nii.gz")))}')
log(f'labelsTr: {len(list((DATASET/"labelsTr").glob("*.nii.gz")))}')
log(f'dataset.json: {(DATASET/"dataset.json").exists()}')
log(f'splits_final.json: {(DATASET/"splits_final.json").exists()}')
log('NCCT BASELINE PREPARATION COMPLETE')


## Recovery rule

If this notebook stops, open it again and **Run all**. Completed 10-subject extraction batches and already copied nnU-Net files are reused. The persistent log is in `MyDrive/HybridStrokeSeg/ISLES2024/results/ncct_prepare.log`.
